# 03 ? Five-model baseline comparison with GroupKFold

**Owner:** Person B ? Modelling Lead

Compare five classifiers using training trips only. Splitting is performed by `trip_id` before sampling, so no trip can occur in both sides of a fold.

## Data and sampling policy

- Only labels 0 and 1 are used for supervised training and validation.
- Label -1 is unlabelled and excluded.
- Features: `perp_dist`, `proj_frac`.
- Five GroupKFold splits.
- Training is capped at 5,000 rows per class per fold.
- Validation is stratified-sampled to at most 20,000 rows per fold, preserving the fold's observed class proportions as closely as possible.
- The same sampled rows are used for all five models in a fold.

**Interpretation:** Metrics estimate performance on sampled validation candidates, not the entire validation fold. This sampling is used to keep the five-model comparison feasible on the available machine. The student-test partition is not loaded or used here.


In [ ]:
from pathlib import Path
import json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, confusion_matrix
)

SEED = 42
MAX_TRAIN_PER_CLASS = 5000
MAX_VALIDATION_ROWS = 20000

TRAIN_CSV = Path("data/processed/train_candidates.csv")
MANIFEST = Path("configs/person_b_feature_manifest.json")
OUTPUT = Path("experiments/results/person_b")
OUTPUT.mkdir(parents=True, exist_ok=True)

if not TRAIN_CSV.exists() or not MANIFEST.exists():
    raise FileNotFoundError("Run notebook 02 first.")

manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
FEATURES = manifest["feature_columns"]
TARGET = manifest["target_column"]
GROUP = manifest["group_column"]

if FEATURES != ["perp_dist", "proj_frac"]:
    raise ValueError(f"Unexpected feature manifest: {FEATURES}")

df = pd.read_csv(TRAIN_CSV)
df = df[df[TARGET].isin([0, 1])].copy()
df[TARGET] = df[TARGET].astype("int8")

if df[TARGET].nunique() != 2:
    raise ValueError("Both binary classes are required.")

if df[FEATURES].isna().any().any():
    raise ValueError("Missing feature values found.")

X = df[FEATURES].astype("float32")
y = df[TARGET].astype("int8")
groups = df[GROUP]

n_splits = min(5, groups.nunique())
if n_splits != 5:
    raise ValueError(f"Expected five folds; found {n_splits} possible folds.")

cv = GroupKFold(n_splits=n_splits)
print(f"Labelled rows: {len(df):,}")
print(f"Trips: {groups.nunique()}")
print("Class counts:", y.value_counts().sort_index().to_dict())
print("Features:", FEATURES)
print("Folds:", n_splits)


In [ ]:
def build_models(seed=SEED):
    return {
        "logistic_regression": make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(
                max_iter=1000, class_weight="balanced",
                random_state=seed, solver="lbfgs"
            )
        ),
        "knn": make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            KNeighborsClassifier(
                n_neighbors=15, weights="distance", n_jobs=2
            )
        ),
        "decision_tree": make_pipeline(
            SimpleImputer(strategy="median"),
            DecisionTreeClassifier(
                max_depth=14, min_samples_leaf=5,
                class_weight="balanced", random_state=seed
            )
        ),
        "random_forest": make_pipeline(
            SimpleImputer(strategy="median"),
            RandomForestClassifier(
                n_estimators=120, min_samples_leaf=2,
                class_weight="balanced_subsample",
                n_jobs=2, random_state=seed
            )
        ),
        "hist_gradient_boosting": make_pipeline(
            SimpleImputer(strategy="median"),
            HistGradientBoostingClassifier(
                max_iter=120, learning_rate=0.08,
                max_leaf_nodes=31, random_state=seed
            )
        )
    }

models = build_models()
print("Models:", list(models))


In [ ]:
def sample_training_indices(indices, labels, cap, seed):
    rng = np.random.default_rng(seed)
    selected = []
    for cls in [0, 1]:
        class_idx = indices[labels.iloc[indices].to_numpy() == cls]
        if len(class_idx) == 0:
            raise ValueError(f"Training fold has no class {cls}")
        take = min(cap, len(class_idx))
        selected.extend(rng.choice(class_idx, size=take, replace=False).tolist())
    rng.shuffle(selected)
    return np.asarray(selected, dtype=int)

def sample_validation_indices(indices, labels, cap, seed):
    if len(indices) <= cap:
        return np.asarray(indices, dtype=int)
    local_y = labels.iloc[indices]
    chosen, _ = train_test_split(
        np.asarray(indices),
        train_size=cap,
        stratify=local_y.to_numpy(),
        random_state=seed
    )
    return np.asarray(chosen, dtype=int)

fold_rows = []
split_audit = []

for fold, (tr_all, va_all) in enumerate(cv.split(X, y, groups), start=1):
    tr_idx = sample_training_indices(
        tr_all, y, MAX_TRAIN_PER_CLASS, SEED + fold
    )
    va_idx = sample_validation_indices(
        va_all, y, MAX_VALIDATION_ROWS, SEED + 100 + fold
    )

    train_trips = set(groups.iloc[tr_idx].unique())
    validation_trips = set(groups.iloc[va_all].unique())
    if train_trips.intersection(validation_trips):
        raise RuntimeError("Trip leakage detected in GroupKFold split")

    if set(y.iloc[tr_idx].unique()) != {0, 1}:
        raise RuntimeError("Sampled training fold lost a class")
    if set(y.iloc[va_idx].unique()) != {0, 1}:
        raise RuntimeError("Sampled validation fold lost a class")

    split_audit.append({
        "fold": fold,
        "train_rows_full": len(tr_all),
        "train_rows_sampled": len(tr_idx),
        "validation_rows_full": len(va_all),
        "validation_rows_sampled": len(va_idx),
        "train_trips": len(train_trips),
        "validation_trips": len(validation_trips),
        "train_class_0": int((y.iloc[tr_idx] == 0).sum()),
        "train_class_1": int((y.iloc[tr_idx] == 1).sum()),
        "validation_class_0": int((y.iloc[va_idx] == 0).sum()),
        "validation_class_1": int((y.iloc[va_idx] == 1).sum()),
        "trip_overlap": 0
    })

    print(
        f"Fold {fold}: train {len(tr_idx):,}/{len(tr_all):,}; "
        f"validation {len(va_idx):,}/{len(va_all):,}; "
        f"trips train={len(train_trips)}, validation={len(validation_trips)}"
    )

    X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
    X_valid, y_valid = X.iloc[va_idx], y.iloc[va_idx]

    for model_name, model in models.items():
        print(f"  Fold {fold} ? {model_name}", flush=True)
        estimator = clone(model)

        start = time.perf_counter()
        estimator.fit(X_train, y_train)
        train_seconds = time.perf_counter() - start

        start = time.perf_counter()
        pred = estimator.predict(X_valid)
        inference_seconds = time.perf_counter() - start

        proba = estimator.predict_proba(X_valid)
        classes = list(estimator.classes_) if hasattr(estimator, "classes_") else [0, 1]
        if classes != [0, 1]:
            raise RuntimeError(f"Unexpected probability class order: {classes}")

        fold_rows.append({
            "model": model_name,
            "fold": fold,
            "accuracy": accuracy_score(y_valid, pred),
            "precision_macro": precision_score(y_valid, pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_valid, pred, average="macro", zero_division=0),
            "macro_f1": f1_score(y_valid, pred, average="macro", zero_division=0),
            "positive_precision": precision_score(y_valid, pred, pos_label=1, zero_division=0),
            "positive_recall": recall_score(y_valid, pred, pos_label=1, zero_division=0),
            "log_loss": log_loss(y_valid, proba, labels=[0, 1]),
            "train_seconds": train_seconds,
            "inference_seconds": inference_seconds,
            "validation_rows": len(va_idx),
            "validation_trips": len(validation_trips)
        })

fold_results = pd.DataFrame(fold_rows)
split_audit_df = pd.DataFrame(split_audit)

fold_results.to_csv(OUTPUT / "baseline_groupkfold_folds.csv", index=False)
split_audit_df.to_csv(OUTPUT / "baseline_groupkfold_split_audit.csv", index=False)

summary = fold_results.groupby("model").agg(
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", "std"),
    precision_macro_mean=("precision_macro", "mean"),
    recall_macro_mean=("recall_macro", "mean"),
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_std=("macro_f1", "std"),
    log_loss_mean=("log_loss", "mean"),
    log_loss_std=("log_loss", "std"),
    train_seconds_mean=("train_seconds", "mean"),
    inference_seconds_mean=("inference_seconds", "mean")
).sort_values("macro_f1_mean", ascending=False)

summary.to_csv(OUTPUT / "baseline_model_comparison.csv")
display(summary)
print("Saved fold metrics, split audit and model comparison to", OUTPUT)


In [ ]:
ax = summary["macro_f1_mean"].sort_values().plot(
    kind="barh", figsize=(9, 5)
)
ax.set_xlabel("Mean sampled GroupKFold macro-F1")
ax.set_title("Five-model baseline comparison ? training trips only")
ax.grid(axis="x", alpha=.25)
plt.tight_layout()
plot_path = OUTPUT / "baseline_macro_f1_comparison.png"
plt.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.close()

best_baseline = summary.index[0]
print("Best untuned baseline by mean macro-F1:", best_baseline)
print("Mean macro-F1:", round(float(summary.loc[best_baseline, "macro_f1_mean"]), 5))
print("Mean log loss:", round(float(summary.loc[best_baseline, "log_loss_mean"]), 5))
print("Saved plot:", plot_path)


## Interpretation and limitations

- `GroupKFold` splits by trip, not candidate row.
- Unlabelled rows (`-1`) are excluded from supervised fitting and CV metrics.
- The training rows are capped per class to control memory and runtime.
- Validation is stratified-sampled; therefore these CV metrics are estimates, not full-fold metrics.
- KNN runtime is measured and included rather than silently skipped.
- Macro-F1 treats both classes equally; log loss measures probability quality.
- The student-test partition is intentionally not accessed in this notebook.
- The best baseline is selected by mean macro-F1 only as a starting point. Bayesian optimisation and held-out student-test evaluation remain separate steps.
